# model

> Audio-JEPA context encoder on Modular MAX.
>
> `build` constructs the model on CPU. `compile_model` traces it for one
> batch size and one spectrogram size. The result is one token per patch.


In [ ]:
#| default_exp model


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
from pathlib import Path

import numpy as np
from max.driver import CPU
from max.dtype import DType
from max.experimental.tensor import Tensor, default_device
from max.graph import TensorType

from nada.audio import VisionTransformer
from nada.config import AudioJEPAConfig, pretrained_config
from nada.core import repo_root
from nada.features import pool_tokens, prepare, read_wav, tile_short
from nada.weights import assign_parameters, load_arrays


In [ ]:
#| export
class AudioJEPA(VisionTransformer):
    """Context encoder. Module names match the published `encoder.*` tensors."""

    pass


In [ ]:
#| export
def build(config: AudioJEPAConfig | None = None) -> AudioJEPA:
    """Construct the model on CPU."""
    with default_device(CPU()):
        return AudioJEPA(config or pretrained_config())


In [ ]:
#| export
def compile_model(model: AudioJEPA, *, batch: int, time_bins: int | None = None, n_mels: int | None = None):
    """Compile the forward pass for `batch` spectrograms of a fixed size.

    Defaults to the pretrained 256 × 128 grid. Recompile when the grid changes.
    """
    time_bins = time_bins or model.config.time_bins
    n_mels = n_mels or model.config.n_mels
    if time_bins % model.config.patch_h or n_mels % model.config.patch_w:
        raise ValueError(f"{time_bins}×{n_mels} does not divide into {model.config.patch_size}")
    return model.compile(
        TensorType(DType.float32, [batch, time_bins, n_mels, model.config.in_chans], device=model.device)
    )


In [ ]:
#| export
def load_checkpoint(model: AudioJEPA, path=None) -> None:
    """Load `models/audio_jepa.safetensors`, or `path` when given."""
    checkpoint = path or (repo_root() / "models" / "audio_jepa.safetensors")
    assign_parameters(model, load_arrays(checkpoint))


In [ ]:
#| export
def to_device(model: AudioJEPA, array: np.ndarray) -> Tensor:
    """Put `array` where the model lives. The default device may be a GPU the model is not on."""
    with default_device(model.device):
        return Tensor(np.ascontiguousarray(array))


In [ ]:
#| export
def center_path(root=None) -> Path:
    """`models/embed_center.npy` next to the checkpoint."""
    return (root or repo_root()) / "models" / "embed_center.npy"


def load_center(path=None) -> np.ndarray | None:
    """Typical-audio vector, or `None` when the file is missing."""
    file = Path(path) if path else center_path()
    if not file.is_file():
        return None
    return np.ascontiguousarray(np.load(file), dtype=np.float32).reshape(-1)


def _synth_clips(sample_rate: int) -> list[np.ndarray]:
    """Tones, clicks, and noise. Not the sounds the app is taught."""
    rng = np.random.default_rng(0)
    clips = []
    for freq, seconds in ((220, 0.45), (330, 0.45), (550, 0.45), (660, 0.45),
                          (990, 0.45), (1200, 0.45), (1600, 0.45), (2400, 0.45),
                          (300, 1.2), (700, 1.2)):
        t = np.arange(int(sample_rate * seconds), dtype=np.float32) / sample_rate
        clips.append((0.2 * np.sin(2 * np.pi * freq * t)).astype(np.float32))
    for seed, seconds, kind in [(i, 0.35, "click") for i in range(6)] + [(i, 0.4, "noise") for i in range(6)]:
        n = int(sample_rate * seconds)
        if kind == "click":
            wave = np.zeros(n, dtype=np.float32)
            k = max(int(sample_rate * 0.02), 1)
            wave[:k] = rng.standard_normal(k).astype(np.float32) * 0.4
            clips.append(wave)
        else:
            clips.append((rng.standard_normal(n) * 0.15).astype(np.float32))
    return clips


def make_center(model: AudioJEPA, compiled, path=None) -> np.ndarray:
    """Mean pooled embedding of the synthetic set. Writes `embed_center.npy`."""
    vectors = [embed(model, compiled, [wave], center=False)[0] for wave in _synth_clips(model.config.sample_rate)]
    center = np.mean(vectors, axis=0).astype(np.float32)
    output = Path(path) if path else center_path()
    output.parent.mkdir(parents=True, exist_ok=True)
    np.save(output, center)
    print(f"wrote {output} from {len(vectors)} clips")
    return center

`embed` tiles a short clip across the 10 s canvas, mean-pools the time rows that have sound, then subtracts `models/embed_center.npy` so short clips do not all point the same way.


In [ ]:
#| export
def embed(model: AudioJEPA, compiled, waveforms, *, pool: bool = True, center: bool = True) -> np.ndarray:
    """Tokens `(batch, patches, dim)`, or one vector per clip when `pool`.

    Short clips are tiled to fill the 10 s canvas, then only time rows with
    sound are averaged. When `center`, subtract `models/embed_center.npy` so
    typical-audio directions cancel. Pass `pool=False` to get every token.
    """
    config = model.config
    if isinstance(waveforms, np.ndarray) and waveforms.ndim == 1:
        waveforms = [waveforms]
    clips = [tile_short(wave, config) for wave in waveforms]
    values = prepare(clips, config)
    tokens = np.asarray(compiled(to_device(model, values)).to_numpy())
    if not pool:
        return tokens
    vectors = pool_tokens(tokens, values, config)
    if center and (typical := load_center()) is not None:
        vectors = vectors - typical
    return vectors


In [ ]:
#| export
def _main() -> None:
    import sys

    if len(sys.argv) < 2:
        raise SystemExit("usage: python -m nada.model clip.wav [more.wav ...]")
    clips = [read_wav(path) for path in sys.argv[1:]]
    model = build()
    load_checkpoint(model)
    compiled = compile_model(model, batch=len(clips))
    vectors = embed(model, compiled, clips)
    for path, vector in zip(sys.argv[1:], vectors, strict=True):
        print(f"{path}: shape={vector.shape} l2={float(np.linalg.norm(vector)):.4f}")


In [ ]:
#| export
#| eval: false
if __name__ == "__main__":
    _main()
